# 02 · Multi-model panel discussion

Several models (or personas) discussing one topic. Two ways to represent a multi-party conversation:

1. **Role mapping:** each participant sees itself as `assistant` and everyone else as `user`.
   That works for two parties but gets awkward beyond that.
2. **Shared transcript:** each turn, the participant gets its persona as the system prompt and the whole
   transcript as one user message. This scales to N participants and works with any provider.

This notebook uses (2) for an architecture review panel.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
from dataclasses import dataclass

from toolkit import chat
from toolkit.notebook import show


@dataclass
class Panelist:
    name: str
    persona: str
    provider: str = "ollama"
    model: str | None = None


TOPIC = (
    "We're a 12-engineer startup with a Django monolith. Should we move to microservices this year?"
)

panel = [
    Panelist(
        "Staff Engineer",
        "Pragmatic and sceptical of rewrites. Cares about delivery speed and operational load.",
    ),
    Panelist(
        "Security Lead",
        "Focuses on attack surface, secrets management, auth between services and compliance.",
    ),
    Panelist(
        "Product Manager",
        "Cares about roadmap risk, time-to-market and what customers actually notice.",
    ),
]

In [ ]:
def turn(speaker: Panelist, transcript: list[tuple[str, str]]) -> str:
    others = ", ".join(p.name for p in panel if p is not speaker)
    system = (
        f"You are the {speaker.name} on an architecture review panel with: {others}. "
        f"{speaker.persona} Respond to the latest points directly, "
        "agree or disagree with specifics, and keep it to 2-4 sentences. "
        "Do not prefix your reply with your name."
    )
    history = "\n\n".join(f"{name}: {text}" for name, text in transcript)
    history = history or "(you open the discussion)"
    user = f"Topic: {TOPIC}\n\nDiscussion so far:\n{history}\n\nYour turn."
    return chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        provider=speaker.provider,
        model=speaker.model,
    )


transcript: list[tuple[str, str]] = []
for _ in range(2):
    for speaker in panel:
        reply = turn(speaker, transcript)
        transcript.append((speaker.name, reply))
        show(f"**{speaker.name}:** {reply}")

### Closing summary by a neutral moderator

In [ ]:
moderator = Panelist(
    "Moderator", "Neutral. Summarize the positions and propose a concrete decision with conditions."
)
show(f"**Moderator:** {turn(moderator, transcript)}")

Notes:
- Mixing providers per panelist (`Panelist(..., provider="openai")`) makes differences in model
  "personality" obvious. Smaller models drift toward agreement within a couple of rounds.
- The transcript grows every turn. For longer sessions it would need summarizing (see week 1, notebook 03).